In [11]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]


In [1]:
import pandas as pd
import requests
import time
import warnings
warnings.filterwarnings("ignore")
 
import pybaseball
pybaseball.cache.enable()
 
from pybaseball import lahman   # built-in dataset, no web scraping needed
 
YEARS = list(range(1980, 2025))
 
print("✅ Imports done")


✅ Imports done


In [22]:
import pandas as pd
import io
import requests

print("Downloading Lahman Teams data...")

url = "https://raw.githubusercontent.com/cbwinslow/baseballdatabank/master/core/Teams.csv"
r = requests.get(url)
teams_df = pd.read_csv(io.StringIO(r.text))

print(f"✅ Teams table: {len(teams_df)} rows")
print(f"Most recent season: {teams_df['yearID'].max()}")
print(f"Columns: {list(teams_df.columns)}")

✅ Teams table: 2985 rows
Most recent season: 2021
Columns: ['yearID', 'lgID', 'teamID', 'franchID', 'divID', 'Rank', 'G', 'Ghome', 'W', 'L', 'DivWin', 'WCWin', 'LgWin', 'WSWin', 'R', 'AB', 'H', '2B', '3B', 'HR', 'BB', 'SO', 'SB', 'CS', 'HBP', 'SF', 'RA', 'ER', 'ERA', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'HRA', 'BBA', 'SOA', 'E', 'DP', 'FP', 'name', 'park', 'attendance', 'BPF', 'PPF', 'teamIDBR', 'teamIDlahman45', 'teamIDretro']


In [23]:
# CELL 4 — League-wide trends

teams = teams_df[
    (teams_df["yearID"] >= 1980) &
    (teams_df["lgID"].isin(["NL", "AL"]))
].copy()

league = teams.groupby("yearID").agg(
    total_G      = ("G",      "sum"),
    total_R      = ("R",      "sum"),
    total_H      = ("H",      "sum"),
    total_HR     = ("HR",     "sum"),
    total_SO     = ("SO",     "sum"),
    total_BB     = ("BB",     "sum"),
    total_AB     = ("AB",     "sum"),
    total_ER     = ("ER",     "sum"),
    total_IPouts = ("IPouts", "sum"),
    total_SOA    = ("SOA",    "sum"),
    total_BBA    = ("BBA",    "sum"),
    total_HRA    = ("HRA",    "sum"),
    num_teams    = ("teamID", "count"),
).reset_index()

league["games"]           = league["total_G"] / 2
league["runs_per_game"]   = round(league["total_R"]   / league["games"], 2)
league["hr_per_game"]     = round(league["total_HR"]  / league["games"], 2)
league["so_per_game"]     = round(league["total_SO"]  / league["games"], 2)
league["bb_per_game"]     = round(league["total_BB"]  / league["games"], 2)
league["batting_avg"]     = round(league["total_H"]   / league["total_AB"], 3)
league["era"]             = round((league["total_ER"] * 27) / league["total_IPouts"], 2)
league["pitcher_k_per9"]  = round((league["total_SOA"] * 27) / league["total_IPouts"], 2)
league["pitcher_bb_per9"] = round((league["total_BBA"] * 27) / league["total_IPouts"], 2)
league["pitcher_hr_per9"] = round((league["total_HRA"] * 27) / league["total_IPouts"], 2)

def get_era_label(year):
    if year <= 1993:   return "Early Modern (1980–1993)"
    elif year <= 2000: return "Steroid Era (1994–2000)"
    elif year <= 2005: return "Steroid Era Peak (2001–2005)"
    elif year <= 2012: return "Pitching Dominance (2006–2012)"
    elif year <= 2019: return "Three True Outcomes (2013–2019)"
    else:              return "Modern Era (2020–present)"

league["era_label"] = league["yearID"].apply(get_era_label)
league["source"]    = "Lahman Database (cbwinslow/baseballdatabank)"

df_league = league[[
    "yearID", "batting_avg", "runs_per_game", "hr_per_game",
    "so_per_game", "bb_per_game", "era",
    "pitcher_k_per9", "pitcher_bb_per9", "pitcher_hr_per9",
    "num_teams", "era_label", "source"
]].rename(columns={"yearID": "season"})

print(df_league[["season", "batting_avg", "runs_per_game", "hr_per_game", "so_per_game", "era"]].to_string())
print(f"\n✅ League trends: {len(df_league)} seasons")

    season  batting_avg  runs_per_game  hr_per_game  so_per_game   era
0     1980        0.265           8.58         1.47         9.60  3.83
1     1981        0.256           8.00         1.28         9.50  3.58
2     1982        0.261           8.60         1.60        10.07  3.85
3     1983        0.261           8.62         1.57        10.30  3.86
4     1984        0.260           8.51         1.55        10.69  3.81
5     1985        0.257           8.66         1.71        10.68  3.89
6     1986        0.258           8.82         1.81        11.75  3.96
7     1987        0.263           9.45         2.12        11.92  4.28
8     1988        0.254           8.28         1.51        11.12  3.72
9     1989        0.254           8.26         1.46        11.23  3.70
10    1990        0.258           8.51         1.58        11.33  3.85
11    1991        0.256           8.62         1.61        11.59  3.91
12    1992        0.256           8.23         1.44        11.18  3.74
13    

In [24]:
# CELL 5 — Team stats by season

team_bat = teams_df[
    (teams_df["yearID"] >= 1980) &
    (teams_df["lgID"].isin(["NL", "AL"]))
].copy()

team_bat["batting_avg"]   = round(team_bat["H"]  / team_bat["AB"], 3)
team_bat["hr_per_game"]   = round(team_bat["HR"] / team_bat["G"],  2)
team_bat["so_per_game"]   = round(team_bat["SO"] / team_bat["G"],  2)
team_bat["bb_per_game"]   = round(team_bat["BB"] / team_bat["G"],  2)
team_bat["runs_per_game"] = round(team_bat["R"]  / team_bat["G"],  2)
team_bat["win_pct"]       = round(team_bat["W"]  / team_bat["G"],  3)
team_bat["era_label"]     = team_bat["yearID"].apply(get_era_label)

df_team = team_bat[[
    "yearID", "teamID", "lgID", "franchID", "name",
    "G", "W", "L", "win_pct",
    "batting_avg", "runs_per_game", "hr_per_game", "so_per_game", "bb_per_game",
    "R", "HR", "SO", "BB", "SB", "AB", "H",
    "ERA", "RA", "ER", "attendance", "era_label"
]].rename(columns={
    "yearID": "season", "teamID": "team_id",
    "lgID": "league", "franchID": "franchise",
    "name": "team_name", "G": "games",
    "W": "wins", "L": "losses",
    "ERA": "team_era", "RA": "runs_allowed",
    "HR": "home_runs", "SO": "strikeouts",
    "BB": "walks", "SB": "stolen_bases",
})

print(f"✅ Team stats: {len(df_team)} team-seasons")
df_team.head()

✅ Team stats: 1198 team-seasons


,season,team_id,league,franchise,team_name,games,wins,losses,win_pct,batting_avg,...,strikeouts,walks,stolen_bases,AB,H,team_era,runs_allowed,ER,attendance,era_label
1787,1980,ATL,NL,ATL,Atlanta Braves,161,81,80,0.503,0.250,...,899.0,434.0,73.0,5402,1352,3.77,660,598,1048411.0,Early Modern (1980–1993)
1788,1980,BAL,AL,BAL,Baltimore Orioles,162,100,62,0.617,0.273,...,766.0,587.0,111.0,5585,1523,3.64,640,591,1797438.0,Early Modern (1980–1993)
1789,1980,BOS,AL,BOS,Boston Red Sox,160,83,77,0.519,0.283,...,720.0,475.0,79.0,5603,1588,4.38,767,701,1956092.0,Early Modern (1980–1993)
1790,1980,CAL,AL,ANA,California Angels,160,65,95,0.406,0.265,...,889.0,539.0,91.0,5443,1442,4.52,797,717,2297327.0,Early Modern (1980–1993)
1791,1980,CHA,AL,CHW,Chicago White Sox,162,70,90,0.432,0.259,...,670.0,399.0,68.0,5444,1408,3.92,722,625,1200365.0,Early Modern (1980–1993)


In [25]:
# CELL 6 — Save to Excel

output_file = "baseball_dashboard1_data.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_league.to_excel(writer, sheet_name="League Trends", index=False)
    df_team.to_excel(writer,   sheet_name="Team Stats by Season", index=False)

    dd = pd.DataFrame([
        ("League Trends", "season",          "Year the season was played"),
        ("League Trends", "batting_avg",      "MLB-wide batting average"),
        ("League Trends", "runs_per_game",    "Runs per game per team"),
        ("League Trends", "hr_per_game",      "Home runs per game per team"),
        ("League Trends", "so_per_game",      "Batter strikeouts per game per team"),
        ("League Trends", "bb_per_game",      "Walks per game per team"),
        ("League Trends", "era",              "MLB-wide ERA"),
        ("League Trends", "pitcher_k_per9",   "Pitcher strikeouts per 9 innings"),
        ("League Trends", "pitcher_bb_per9",  "Pitcher walks per 9 innings"),
        ("League Trends", "pitcher_hr_per9",  "HR allowed per 9 innings"),
        ("League Trends", "era_label",        "Era name — use for color/filter in Tableau"),
        ("League Trends", "source",           "Data source"),
        ("Team Stats by Season", "season",    "Year"),
        ("Team Stats by Season", "team_name", "Full team name"),
        ("Team Stats by Season", "league",    "AL or NL"),
        ("Team Stats by Season", "win_pct",   "Win percentage"),
        ("Team Stats by Season", "team_era",  "Team ERA"),
        ("Team Stats by Season", "batting_avg","Team batting average"),
        ("Team Stats by Season", "attendance","Season attendance"),
    ], columns=["sheet", "field", "description"])
    dd.to_excel(writer, sheet_name="Data Dictionary", index=False)

print(f"✅ Saved: {output_file}")
print(f"\n  League Trends        — {len(df_league)} seasons (1980–2021)")
print(f"  Team Stats by Season — {len(df_team)} team-seasons")
print("\nReady to connect in Tableau!")

✅ Saved: baseball_dashboard1_data.xlsx

  League Trends        — 42 seasons (1980–2021)
  Team Stats by Season — 1198 team-seasons

Ready to connect in Tableau!
